# Visualization
Paper-quality figures for the robust CFLP study.  
Edit the **Configuration** cell, then run the figure cells you need.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
EXCEL = "05_sensitivity_results.xlsx"   # relative to experiments/, or absolute path

# Slice used by single-panel figures (Fig 1, Fig 3)
V_SCALE = 0.75
W_VAL   = 10
GAMMAS  = [1, 2, 3, 4]

# Full parameter grid (used by multi-panel figures)
V_LIST = [0.75, 1.0, 1.25]
W_LIST = [10, 100, 1000]

In [ ]:
import importlib, sys, pathlib

_here = pathlib.Path().resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import fig_violin_cvar as fvc
importlib.reload(fvc)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.lines as mlines

%matplotlib inline
plt.rcParams.update(fvc.mpl.rcParams)

# ── Shared colour / marker scheme for v values ────────────────────────────────
V_COLORS  = {0.75: "#1b7837", 1.0: "#762a83", 1.25: "#d6604d"}
V_MARKERS = {0.75: "o",        1.0: "s",        1.25: "^"}
V_LABELS  = {0.75: "$v=0.75$", 1.0: "$v=1.00$", 1.25: "$v=1.25$"}
W_TITLES  = {10:   "$w=10$ (low congestion)",
             100:  "$w=100$ (moderate)",
             1000: "$w=1000$ (high congestion)"}

# ── Colours for adaptive vs fixed-price violins ───────────────────────────────
ADV_LIGHT = "#74c476"   # adaptive pricing — light green
ADV_DARK  = "#238b45"   # adaptive pricing — dark green
FIX_LIGHT = "#fdae6b"   # fixed pricing    — light orange
FIX_DARK  = "#d94801"   # fixed pricing    — dark orange

In [ ]:
oos  = pd.read_excel(EXCEL, sheet_name="OOS_Raw")
sens = pd.read_excel(EXCEL, sheet_name="Summary")
print(f"OOS rows : {len(oos)}  |  columns: {list(oos.columns)}")
print(f"Summary  : {len(sens)} rows")

---
## Fig 1 — Violin + CVaR 5%: Nominal vs Robust (OOS profit distribution)
One pair of violins per Γ. Dark tail = bottom 5% of the distribution; bold bar = CVaR₅%.

In [ ]:
HALF_W  = 0.32
GAP     = 0.08
GROUP_W = 2.0

sub = oos[(oos["v_scale"] == V_SCALE) & (oos["w"] == W_VAL)]

fig, ax = plt.subplots(figsize=(10, 5.5))
x_ticks, x_labels, cvar_summary = [], [], {}

for gi, gam in enumerate(GAMMAS):
    g = sub[sub["gamma"] == gam]
    x_base = gi * GROUP_W
    x_nom  = x_base - (HALF_W + GAP / 2)
    x_rob  = x_base + (HALF_W + GAP / 2)

    cv_nom = fvc.draw_violin(ax, g["profit_a_nom"].values, x_nom, HALF_W,
                             fvc.NOM_LIGHT, fvc.NOM_DARK,
                             label="Nominal $x$" if gi == 0 else None)
    cv_rob = fvc.draw_violin(ax, g["profit_a_rob"].values, x_rob, HALF_W,
                             fvc.ROB_LIGHT, fvc.ROB_DARK,
                             label="Robust $x$" if gi == 0 else None)
    cvar_summary[gam] = (cv_nom, cv_rob)
    x_ticks.append(x_base)
    x_labels.append(f"$\\Gamma={gam}$")

ax.axhline(0, color="black", lw=0.9, ls=":", zorder=1, alpha=0.6)
ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels)
ax.set_xlim(x_ticks[0] - GROUP_W*0.7, x_ticks[-1] + GROUP_W*0.7)
ax.set_ylabel("Out-of-sample profit")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.grid(axis="y", ls="--")
ax.legend(handles=[
    mpatches.Patch(color=fvc.NOM_LIGHT, alpha=0.7,  label="Nominal $x$  (Scen. A)"),
    mpatches.Patch(color=fvc.ROB_LIGHT, alpha=0.7,  label="Robust $x$   (Scen. A)"),
    mpatches.Patch(color=fvc.NOM_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Nominal"),
    mpatches.Patch(color=fvc.ROB_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Robust"),
    plt.Line2D([0],[0], color="gray", lw=2.2, label="Median"),
    plt.Line2D([0],[0], color="gray", lw=1.2, ls="--", label="5th percentile (VaR)"),
    plt.Line2D([0],[0], marker="D", color="w", markeredgecolor="gray", ms=5, label="Mean"),
], ncol=2, frameon=True, framealpha=0.92, loc="upper right", fontsize=8.5, edgecolor="0.8")
ax.set_title(f"OOS Profit — Nominal vs. Robust  ($v={V_SCALE},\\;w={W_VAL}$, adaptive pricing)",
             fontsize=11, pad=8)
fig.tight_layout()
fig.savefig("fig1_violin_cvar.pdf", bbox_inches="tight")
fig.savefig("fig1_violin_cvar.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\n{'Γ':>4}  {'CVaR5 Nom':>12}  {'CVaR5 Rob':>12}  {'Δ CVaR5':>12}")
print("-" * 46)
for gam, (cn, cr) in cvar_summary.items():
    print(f"{gam:>4}  {cn:>12,.0f}  {cr:>12,.0f}  {cr-cn:>+12,.0f}")

---
## Fig 2 — Price of Robustness (PoR) & Value of Robustification (VoR)

**PoR** = profit sacrificed by the robust design when no disruption occurs:  
$\text{PoR}(\%) = 100 \times (\pi^{\text{nom}}_0 - \pi^{\text{rob}}_0)\;/\;|\pi^{\text{nom}}_0|$

**VoR** = extra profit from the robust design under the worst-case disruption:  
$\text{VoR} = \pi^{\text{rob}}_{\text{wc}} - \pi^{\text{nom}}_{\text{wc}}$

Layout: 2 rows × 3 columns (one column per *w*). Lines coloured by *v*.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey="row")

for ci, w in enumerate(W_LIST):
    ax_por, ax_vor = axes[0, ci], axes[1, ci]
    for v in V_LIST:
        sub = sens[(sens["v_scale"] == v) & (sens["w"] == w)].sort_values("gamma")
        g   = sub["gamma"].values
        por = 100 * (sub["nd_nom"].values - sub["nd_rob"].values) / np.abs(sub["nd_nom"].values)
        vor = sub["wc_rob_val_a"].values
        kw  = dict(color=V_COLORS[v], marker=V_MARKERS[v], markersize=6, linewidth=1.6, label=V_LABELS[v])
        ax_por.plot(g, por, **kw)
        ax_vor.plot(g, vor, **kw)

    for ax, ylabel, fmt in [
        (ax_por, "Price of Robustness PoR (%)",    lambda y, _: f"{y:.1f}%"),
        (ax_vor, "Value of Robustification VoR",   lambda y, _: f"${y/1e3:.0f}k"),
    ]:
        ax.axhline(0, color="black", lw=0.7, ls=":", alpha=0.5)
        ax.set_xticks(GAMMAS); ax.set_xticklabels([f"$\\Gamma={g}$" for g in GAMMAS])
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.grid(axis="y", ls="--", alpha=0.4)
        if ci == 0: ax.set_ylabel(ylabel, fontsize=10)

    ax_por.set_title(W_TITLES[w], fontsize=10)

handles = [mlines.Line2D([], [], color=V_COLORS[v], marker=V_MARKERS[v],
                         markersize=6, lw=1.6, label=V_LABELS[v]) for v in V_LIST]
axes[0, 2].legend(handles=handles, frameon=True, framealpha=0.9, fontsize=9, loc="upper left")
fig.suptitle("Cost vs. Benefit of Robustification — PoR (top) and VoR (bottom)",
             fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig("fig2_por_vor.pdf", bbox_inches="tight")
fig.savefig("fig2_por_vor.png", dpi=300, bbox_inches="tight")
plt.show()

---
## Fig 3 — Value of Adaptive Pricing: OOS distribution (Adaptive vs Fixed-price)
Same violin format as Fig 1, but for the **robust solution** comparing adaptive pricing (Scen. A)
vs committed fixed prices (Scen. B). CVaR₅% tails show the downside-risk improvement.

In [ ]:
HALF_W  = 0.32
GAP     = 0.08
GROUP_W = 2.0

sub = oos[(oos["v_scale"] == V_SCALE) & (oos["w"] == W_VAL)]

fig, ax = plt.subplots(figsize=(10, 5.5))
x_ticks, x_labels, cvar_summary = [], [], {}

for gi, gam in enumerate(GAMMAS):
    g = sub[sub["gamma"] == gam]
    x_base = gi * GROUP_W
    x_adv  = x_base - (HALF_W + GAP / 2)
    x_fix  = x_base + (HALF_W + GAP / 2)

    cv_adv = fvc.draw_violin(ax, g["profit_a_rob"].values, x_adv, HALF_W,
                             ADV_LIGHT, ADV_DARK,
                             label="Adaptive pricing" if gi == 0 else None)
    cv_fix = fvc.draw_violin(ax, g["profit_b_rob"].values, x_fix, HALF_W,
                             FIX_LIGHT, FIX_DARK,
                             label="Fixed pricing" if gi == 0 else None)
    cvar_summary[gam] = (cv_adv, cv_fix)
    x_ticks.append(x_base)
    x_labels.append(f"$\\Gamma={gam}$")

ax.axhline(0, color="black", lw=0.9, ls=":", zorder=1, alpha=0.6)
ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels)
ax.set_xlim(x_ticks[0] - GROUP_W*0.7, x_ticks[-1] + GROUP_W*0.7)
ax.set_ylabel("Out-of-sample profit (robust $x$)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.grid(axis="y", ls="--")
ax.legend(handles=[
    mpatches.Patch(color=ADV_LIGHT, alpha=0.7,  label="Adaptive pricing  (Scen. A)"),
    mpatches.Patch(color=FIX_LIGHT, alpha=0.7,  label="Fixed pricing     (Scen. B)"),
    mpatches.Patch(color=ADV_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Adaptive"),
    mpatches.Patch(color=FIX_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Fixed"),
    plt.Line2D([0],[0], color="gray", lw=2.2, label="Median"),
    plt.Line2D([0],[0], color="gray", lw=1.2, ls="--", label="5th percentile (VaR)"),
    plt.Line2D([0],[0], marker="D", color="w", markeredgecolor="gray", ms=5, label="Mean"),
], ncol=2, frameon=True, framealpha=0.92, loc="upper right", fontsize=8.5, edgecolor="0.8")
ax.set_title(
    f"OOS Profit — Adaptive vs. Fixed Pricing (robust $x$, $v={V_SCALE},\\;w={W_VAL}$)",
    fontsize=11, pad=8)
fig.tight_layout()
fig.savefig("fig3_vap_violin.pdf", bbox_inches="tight")
fig.savefig("fig3_vap_violin.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\n{'Γ':>4}  {'CVaR5 Adv':>12}  {'CVaR5 Fix':>12}  {'Δ CVaR5 (VAP)':>14}")
print("-" * 48)
for gam, (ca, cf) in cvar_summary.items():
    print(f"{gam:>4}  {ca:>12,.0f}  {cf:>12,.0f}  {ca-cf:>+14,.0f}")

---
## Fig 4 — VAP sensitivity: average gain and CVaR₅% improvement

**Top row**: VAP (%) = average out-of-sample profit uplift from adaptive pricing for the robust solution  
$\text{VAP}(\%) = 100 \times (\bar\pi^A_{\text{rob}} - \bar\pi^B_{\text{rob}})\;/\;|\bar\pi^B_{\text{rob}}|$

**Bottom row**: CVaR₅% improvement (%) = how much adaptive pricing lifts the tail-risk metric  
$\Delta\text{CVaR}_5(\%) = 100 \times (\text{CVaR}^A_{\text{rob}} - \text{CVaR}^B_{\text{rob}})\;/\;|\text{CVaR}^B_{\text{rob}}|$

Higher *v* (more price-sensitive demand) should drive larger gains in both rows.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey="row")

for ci, w in enumerate(W_LIST):
    ax_vap  = axes[0, ci]
    ax_cvar = axes[1, ci]

    for v in V_LIST:
        sub = sens[(sens["v_scale"] == v) & (sens["w"] == w)].sort_values("gamma")
        g   = sub["gamma"].values

        # VAP%: average OOS profit improvement (adaptive vs fixed, robust solution)
        vap_pct = 100 * sub["vap_rob_mean"].values / np.abs(sub["b_rob_mean"].values)

        # ΔCVaR5%: tail-risk improvement
        dcvar_pct = 100 * (sub["a_rob_cvar5"].values - sub["b_rob_cvar5"].values) \
                        / np.abs(sub["b_rob_cvar5"].values)

        kw = dict(color=V_COLORS[v], marker=V_MARKERS[v], markersize=6, lw=1.6, label=V_LABELS[v])
        ax_vap.plot(g, vap_pct, **kw)
        ax_cvar.plot(g, dcvar_pct, **kw)

    for ax, ylabel in [
        (ax_vap,  "VAP — avg. profit uplift (%)"),
        (ax_cvar, "$\\Delta$CVaR$_{5\\%}$ improvement (%)"),
    ]:
        ax.axhline(0, color="black", lw=0.7, ls=":", alpha=0.5)
        ax.set_xticks(GAMMAS); ax.set_xticklabels([f"$\\Gamma={g}$" for g in GAMMAS])
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}%"))
        ax.grid(axis="y", ls="--", alpha=0.4)
        if ci == 0: ax.set_ylabel(ylabel, fontsize=10)

    ax_vap.set_title(W_TITLES[w], fontsize=10)

handles = [mlines.Line2D([], [], color=V_COLORS[v], marker=V_MARKERS[v],
                          markersize=6, lw=1.6, label=V_LABELS[v]) for v in V_LIST]
axes[0, 2].legend(handles=handles, frameon=True, framealpha=0.9, fontsize=9, loc="upper left")
fig.suptitle("Value of Adaptive Pricing — Average Gain (top) and CVaR₅% Improvement (bottom)",
             fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig("fig4_vap_sensitivity.pdf", bbox_inches="tight")
fig.savefig("fig4_vap_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

---
## Fig 5 — Does adaptive pricing cover the cost of robustness?

Each point is one (v, w, Γ) combination.  
**X-axis**: PoR (%) — the profit sacrificed by choosing the robust design under no disruption.  
**Y-axis**: VAP (%) — the average profit recovered by adaptive pricing vs fixed pricing (robust solution).  

The dashed reference line **y = x** marks break-even: points *above* the line mean adaptive
pricing more than compensates for the cost of robustness.

In [ ]:
# ── Marker size encodes Γ ─────────────────────────────────────────────────────
GAMMA_SIZES = {1: 40, 2: 70, 3: 110, 4: 160}

fig, ax = plt.subplots(figsize=(7, 6))

all_x, all_y = [], []

for v in V_LIST:
    for w in W_LIST:
        sub = sens[(sens["v_scale"] == v) & (sens["w"] == w)].sort_values("gamma")
        for _, r in sub.iterrows():
            por  = 100 * (r.nd_nom - r.nd_rob)    / abs(r.nd_nom)
            vap  = 100 * r.vap_rob_mean            / abs(r.b_rob_mean)
            all_x.append(por); all_y.append(vap)
            ax.scatter(por, vap,
                       color=V_COLORS[v], marker=V_MARKERS[v],
                       s=GAMMA_SIZES[int(r.gamma)],
                       edgecolors="white", linewidths=0.5,
                       alpha=0.85, zorder=3)

# Break-even line y = x
lim = max(max(all_x), max(all_y)) * 1.1
ax.plot([0, lim], [0, lim], color="black", lw=1.2, ls="--", alpha=0.5,
        label="Break-even ($y = x$)", zorder=1)
ax.fill_between([0, lim], [0, lim], [lim, lim],
                color="#a1d99b", alpha=0.10, zorder=0,
                label="VAP > PoR (net gain)")
ax.fill_between([0, lim], [0, 0], [0, lim],
                color="#fc9272", alpha=0.10, zorder=0,
                label="VAP < PoR (net loss)")

ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("Price of Robustness — PoR (%)", fontsize=11)
ax.set_ylabel("Value of Adaptive Pricing — VAP (%)", fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}%"))
ax.grid(ls="--", alpha=0.3)

# Legend: v colours
v_handles = [mlines.Line2D([], [], color=V_COLORS[v], marker=V_MARKERS[v],
                            ls="None", markersize=8, label=V_LABELS[v]) for v in V_LIST]
# Legend: Γ sizes
g_handles = [mlines.Line2D([], [], color="gray", marker="o", ls="None",
                            markersize=np.sqrt(GAMMA_SIZES[g]),
                            label=f"$\\Gamma={g}$") for g in GAMMAS]
ax.legend(handles=v_handles + g_handles + [
              mlines.Line2D([0],[0], color="black", lw=1.2, ls="--", label="Break-even")],
          ncol=2, fontsize=8.5, frameon=True, framealpha=0.92, edgecolor="0.8")

ax.set_title("Adaptive Pricing vs. Cost of Robustness\n"
             "(above dashed line: VAP more than offsets PoR)",
             fontsize=11, pad=8)
fig.tight_layout()
fig.savefig("fig5_vap_vs_por.pdf", bbox_inches="tight")
fig.savefig("fig5_vap_vs_por.png", dpi=300, bbox_inches="tight")
plt.show()

total = len(all_x)
above = sum(y > x for x, y in zip(all_x, all_y))
print(f"Points above break-even (VAP > PoR): {above}/{total} ({100*above/total:.0f}%)")